# The Missing Baseline: Does the Real, Undecomposed UCCS Vector Actually Work?

Every prior run in this project skipped straight to steering with an SAE-decomposed minimal
vector. Nobody ever tested the original method (UCCS -- Zhang et al., ICLR 2026,
"Controlling Repetition in Protein Language Models") as-is: just the raw contrastive
difference vector v_L = mean(positive activations) - mean(negative activations), injected
directly, no sparse autoencoder involved.

This notebook runs FOUR conditions in one session, so they're directly comparable:
1. **CONTROL** -- unsteered.
2. **FULL_UCCS_1x** -- raw v_L injected at its natural scale.
3. **FULL_UCCS_2x** -- raw v_L at 2x scale.
4. **OSAE_MINIMAL_2x** -- the same minimal 3-latent OSAE steering tested before, as an
   internal consistency check against the earlier N=200 run.

If FULL_UCCS fixes repetition without breaking structure, the earlier OSAE-decomposition
step is the culprit. If FULL_UCCS fails too, this project's implementation (or this small
model / short-sequence setup) doesn't reproduce the paper's claimed result, independent of
sparse autoencoders entirely.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(42)
np.random.seed(42)

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())

Setup complete. CUDA available: True


In [4]:
class OrderedSAE(nn.Module):
    def __init__(self, d_model=1280, expansion_factor=4, k=16, nesting_list=[64, 256, 1024, 5120]):
        super().__init__()
        self.d_latent = d_model * expansion_factor
        self.k = k
        self.nesting_list = nesting_list
        self.encoder = nn.Linear(d_model, self.d_latent)
        self.decoder = nn.Linear(self.d_latent, d_model)

    def forward(self, x):
        acts = F.relu(self.encoder(x))
        topk_vals, topk_idx = torch.topk(acts, self.k, dim=-1)
        sparse_acts = torch.zeros_like(acts).scatter_(-1, topk_idx, topk_vals)
        reconstructions = []
        for nest_size in self.nesting_list:
            mask = torch.zeros_like(sparse_acts)
            mask[:, :nest_size] = 1.0
            reconstructions.append(self.decoder(sparse_acts * mask))
        return reconstructions, sparse_acts


class PLMExtractor:
    def __init__(self, target_layer=12):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading ProtGPT2 on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
        self.model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(self.device)
        self.target_layer = target_layer

In [5]:
# --- Build v_L (the real UCCS steering direction) and, alongside it, the OSAE decomposition ---

positive_seqs = [
    "NLYIQWLKDGGPSSGRPPPS", "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF", "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]
degenerate_seqs = [
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA", "LGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGL",
    "GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG", "SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS",
    "PGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGP", "QWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQ",
]

def get_mean_activation(extractor, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = extractor.tokenizer(seq, return_tensors="pt").to(extractor.device)
        with torch.no_grad():
            out = extractor.model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

def train_ordered_sae(X_train, device, epochs=1000, lr=1e-3):
    osae = OrderedSAE(d_model=X_train.shape[1], expansion_factor=4, k=16).to(device)
    optimizer = torch.optim.Adam(osae.parameters(), lr=lr)
    X_train = X_train.to(device)
    osae.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        reconstructions, _ = osae(X_train)
        loss = sum(F.mse_loss(recon, X_train) for recon in reconstructions)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 200 == 0:
            print(f"  epoch {epoch+1}/{epochs} | cumulative MRL loss: {loss.item():.4f}")
    return osae

print("=== Extracting v_L and training the OSAE (for the consistency-check condition) ===")
extractor = PLMExtractor(target_layer=12)

pos_acts = get_mean_activation(extractor, positive_seqs, extractor.target_layer)
neg_acts = get_mean_activation(extractor, degenerate_seqs, extractor.target_layer)
v_L = (pos_acts.mean(dim=0) - neg_acts.mean(dim=0)).to(extractor.device)
print(f"v_L norm: {v_L.norm().item():.4f}  (shape {tuple(v_L.shape)})")

X_train_mixed = torch.cat([pos_acts, neg_acts], dim=0).to(extractor.device)
osae_model = train_ordered_sae(X_train_mixed, device=extractor.device)

osae_model.eval()
with torch.no_grad():
    acts = F.relu(osae_model.encoder(v_L.unsqueeze(0)))
    topk_vals, topk_idx = torch.topk(acts, osae_model.k, dim=-1)
    sparse_v_L = torch.zeros_like(acts).scatter_(-1, topk_idx, topk_vals).squeeze(0)

active_indices = torch.nonzero(sparse_v_L).squeeze(-1).cpu().numpy()
active_weights = sparse_v_L[active_indices].detach().cpu().numpy()
order = np.argsort(-np.abs(active_weights))
active_indices, active_weights = active_indices[order], active_weights[order]

TOP_K_CAUSAL = 3
causal_indices = active_indices[:TOP_K_CAUSAL].tolist()
causal_weights = {int(i): float(w) for i, w in zip(active_indices[:TOP_K_CAUSAL], active_weights[:TOP_K_CAUSAL])}
print(f"Top-{TOP_K_CAUSAL} OSAE latents this run: {causal_indices}")

with torch.no_grad():
    mock_latent = torch.zeros(1, 1, osae_model.d_latent, device=extractor.device)
    for idx, w in causal_weights.items():
        mock_latent[:, :, idx] = w * 2.0
    osae_minimal_vector_2x = osae_model.decoder(mock_latent).squeeze(0).squeeze(0)
print(f"OSAE-minimal-2x vector norm: {osae_minimal_vector_2x.norm().item():.4f}")

=== Extracting v_L and training the OSAE (for the consistency-check condition) ===
Loading ProtGPT2 on cuda...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


v_L norm: 583.9979  (shape (1280,))
  epoch 200/1000 | cumulative MRL loss: 12712.1904
  epoch 400/1000 | cumulative MRL loss: 8723.7305
  epoch 600/1000 | cumulative MRL loss: 3030.3601
  epoch 800/1000 | cumulative MRL loss: 628.7200
  epoch 1000/1000 | cumulative MRL loss: 291.2990
Top-3 OSAE latents this run: [3664, 2547, 946]
OSAE-minimal-2x vector norm: 216.4437


In [6]:
# --- Same real, UniProt-derived calibration prefixes as before ---
import urllib.request

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

if len(reference_seqs) < 5:
    raise RuntimeError("Fewer than 5 reference sequences fetched -- check Kaggle internet access is ON.")

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=7):
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

calibration_prefixes = build_prefix_pool(reference_seqs, n_prefixes=200)
print(f"Built {len(calibration_prefixes)} calibration prefixes from {len(reference_seqs)} reference proteins.")

Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  skip P42212: The read operation timed out
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  skip P00648: The read operation timed out
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues
Built 200 calibration prefixes from 10 reference proteins.


In [7]:
# --- Generic vector-injection generator: works for both the raw UCCS vector and the OSAE-minimal vector ---

def generate_with_vector_steering(extractor, steering_vector, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    extractor.model.eval()
    device = next(extractor.model.parameters()).device
    v = None if steering_vector is None else steering_vector.to(device)

    def hook(module, inp, out):
        if v is None:
            return out
        return (out[0] + v,)

    records = []
    for prompt in prompts:
        handle = extractor.model.transformer.h[extractor.target_layer].register_forward_hook(hook)
        inputs = extractor.tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output_ids = extractor.model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=extractor.tokenizer.eos_token_id
            )
        handle.remove()
        seq = extractor.tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_part = seq[len(prompt):] if seq.startswith(prompt) else seq
        records.append({"prompt": prompt, "sequence": seq, "entropy": calculate_entropy(gen_part)})
    clear_gpu()
    return records

print("Vector-steering generator ready.")

Vector-steering generator ready.


In [8]:
N_PER_CONDITION = 50
conditions = {}

print("=== CONTROL (unsteered) ===")
conditions["CONTROL"] = generate_with_vector_steering(
    extractor, None, calibration_prefixes[0:N_PER_CONDITION], seed=101
)

print("=== FULL_UCCS_1x (raw v_L, natural scale) ===")
conditions["FULL_UCCS_1x"] = generate_with_vector_steering(
    extractor, v_L * 1.0, calibration_prefixes[N_PER_CONDITION:2*N_PER_CONDITION], seed=202
)

print("=== FULL_UCCS_2x (raw v_L, 2x scale) ===")
conditions["FULL_UCCS_2x"] = generate_with_vector_steering(
    extractor, v_L * 2.0, calibration_prefixes[2*N_PER_CONDITION:3*N_PER_CONDITION], seed=303
)

print("=== OSAE_MINIMAL_2x (consistency check vs. the earlier N=200 run) ===")
conditions["OSAE_MINIMAL_2x"] = generate_with_vector_steering(
    extractor, osae_minimal_vector_2x, calibration_prefixes[3*N_PER_CONDITION:4*N_PER_CONDITION], seed=404
)

for name, recs in conditions.items():
    print(f"{name}: {len(recs)} sequences generated")

=== CONTROL (unsteered) ===
=== FULL_UCCS_1x (raw v_L, natural scale) ===
=== FULL_UCCS_2x (raw v_L, 2x scale) ===
=== OSAE_MINIMAL_2x (consistency check vs. the earlier N=200 run) ===
CONTROL: 50 sequences generated
FULL_UCCS_1x: 50 sequences generated
FULL_UCCS_2x: 50 sequences generated
OSAE_MINIMAL_2x: 50 sequences generated


In [9]:
print("=== Freeing ProtGPT2 + OSAE from GPU ===")
del extractor
del osae_model
clear_gpu()
print(f"GPU memory allocated after purge: {torch.cuda.memory_allocated()/1e9:.3f} GB")

=== Freeing ProtGPT2 + OSAE from GPU ===
GPU memory allocated after purge: 0.018 GB


In [10]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0
        inputs = self.tokenizer([cleaned], return_tensors="pt", add_special_tokens=False).to(self.device)
        t0 = time.time()
        plddt = 0.0
        try:
            with torch.no_grad():
                out = self.model(**inputs)
            raw = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw * 100.0 if raw <= 1.5 else raw
        except RuntimeError:
            clear_gpu()
        dt = time.time() - t0
        return plddt, dt

evaluator = StructuralEvaluator()

def fold_records(records, evaluator):
    for r in records:
        plddt, dt = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["fold_time_s"] = dt
        r["collapse"] = int(0.0 < plddt < 60.0)
    return records

for name in conditions:
    print(f"Folding {name}...")
    conditions[name] = fold_records(conditions[name], evaluator)

del evaluator
clear_gpu()

Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding FULL_UCCS_1x...
Folding FULL_UCCS_2x...
Folding OSAE_MINIMAL_2x...


In [11]:
# --- The verdict table ---
print(f"{'Condition':18s} {'N':>4s} {'Entropy':>9s} {'pLDDT':>8s} {'Collapse%':>10s}")
print("-" * 55)
for name, recs in conditions.items():
    ents = [r["entropy"] for r in recs]
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0.0]
    collapse_rate = np.mean([r["collapse"] for r in recs])
    mean_plddt = np.mean(plddts) if plddts else float("nan")
    print(f"{name:18s} {len(recs):4d} {np.mean(ents):9.3f} {mean_plddt:8.2f} {collapse_rate*100:9.1f}%")

print()
print("Read this against CONTROL:")
print("- If FULL_UCCS_1x/2x show HIGHER entropy than control with pLDDT close to control's:")
print("  the real UCCS method works in this setup -- the OSAE decomposition step is what")
print("  loses the benefit. That's your paper.")
print("- If FULL_UCCS also fails to raise entropy over control, or also craters pLDDT similarly")
print("  to OSAE_MINIMAL_2x: the method doesn't reproduce here independent of any SAE step --")
print("  a different, still legitimate, paper (replication failure / small-model limits).")
print("- OSAE_MINIMAL_2x here should roughly match the earlier N=200 finding")
print("  (control ~60.4 pLDDT / ~3.50 entropy -> steered ~57.9 pLDDT / ~3.39 entropy)")
print("  as an internal sanity check that this run's environment is behaving consistently.")

Condition             N   Entropy    pLDDT  Collapse%
-------------------------------------------------------
CONTROL              50     2.884    59.47      62.0%
FULL_UCCS_1x         50     2.435    56.25      60.0%
FULL_UCCS_2x         50     4.051    30.76     100.0%
OSAE_MINIMAL_2x      50     2.625    58.56      50.0%

Read this against CONTROL:
- If FULL_UCCS_1x/2x show HIGHER entropy than control with pLDDT close to control's:
  the real UCCS method works in this setup -- the OSAE decomposition step is what
  loses the benefit. That's your paper.
- If FULL_UCCS also fails to raise entropy over control, or also craters pLDDT similarly
  to OSAE_MINIMAL_2x: the method doesn't reproduce here independent of any SAE step --
  a different, still legitimate, paper (replication failure / small-model limits).
- OSAE_MINIMAL_2x here should roughly match the earlier N=200 finding
  (control ~60.4 pLDDT / ~3.50 entropy -> steered ~57.9 pLDDT / ~3.39 entropy)
  as an internal sanity check t